In [1]:
import numpy as np
from efficient_probit_regression.sampling import calculate_lewis_weights_exact, compute_leverage_scores, to_density

# Laden der notwendigen Datensaetze
%store -r Iris
%store -r Webspam
%store -r KDDCup
%store -r Covertype
%store -r Example2D

print("Bestimmung von c = max(np.maximum(lewis_prob / scaled_l2_lp_prob, scaled_l2_lp_prob / lewis_prob))")
print("")

p = [2.3, 2.5, 3.0, 3.5, 4.0]
LEWIS_ITERATIONS = 20
EPS = 1e-300
rows_c = []

datasets = {
    "Iris": Iris,
    "Example2D": Example2D,
    "Webspam": Webspam,
    "Covertype": Covertype,
    "KDDCup": KDDCup,
}


def scale_factor_d_p_over_2_minus_1(d: int, p_val: float):
    return float(d ** (p_val / 2.0 - 1.0))


def calculate_scaled_l2_lp_scores(X, p_val):
    d = X.shape[1]
    scale_factor = scale_factor_d_p_over_2_minus_1(d, p_val)
    l2_scores = compute_leverage_scores(X, p=2.0, fast_approx=False)
    lp_scores = compute_leverage_scores(X, p=p_val, fast_approx=False)
    scaled_scores = scale_factor * l2_scores + lp_scores
    scaled_prob = to_density(scaled_scores)
    return scaled_scores, scaled_prob, d, scale_factor


for p_val in p:
    print("p = ", p_val)
    print("")

    for dataset_name, ds in datasets.items():
        print(f"{dataset_name} Datensatz")
        lewis_weights = calculate_lewis_weights_exact(ds.X, p=p_val, T=LEWIS_ITERATIONS)
        lewis_prob = to_density(lewis_weights)
        scaled_scores, scaled_prob, d, scale_factor = calculate_scaled_l2_lp_scores(ds.X, p_val)

        c_value = float(np.max(np.maximum(lewis_prob / np.maximum(scaled_prob, EPS), scaled_prob / np.maximum(lewis_prob, EPS))))
        print("c:", c_value)
        rows_c.append({
            "dataset": dataset_name,
            "p": float(p_val),
            "d": int(d),
            "scale_factor": float(scale_factor),
            "c": float(c_value),
        })
        print("")


Bestimmung von c = max(np.maximum(lewis_prob / scaled_l2_lp_prob, scaled_l2_lp_prob / lewis_prob))

p =  2.3

Iris Datensatz
c: 1.207028215021254

Example2D Datensatz
c: 1.1586448179326796

Webspam Datensatz
c: 2.304970051788473

Covertype Datensatz
c: 62.615982039287225

KDDCup Datensatz
c: 2.086717261880734

p =  2.5

Iris Datensatz
c: 1.372645308946349

Example2D Datensatz
c: 1.2763343623359125

Webspam Datensatz
c: 3.8180802969606535

Covertype Datensatz
c: 578.3220636546915

KDDCup Datensatz
c: 3.235341501577226

p =  3.0

Iris Datensatz
c: 1.8946899816614395

Example2D Datensatz
c: 1.5952135478092349

Webspam Datensatz
c: 10.941407424610402

Covertype Datensatz
c: 2055.236710064572

KDDCup Datensatz
c: 8.069251496159872

p =  3.5

Iris Datensatz
c: 2.623346917228983

Example2D Datensatz
c: 1.9873942733250463

Webspam Datensatz
c: 23.745769043305945

Covertype Datensatz
c: 2096.9813680631123

KDDCup Datensatz
c: 15.23334476668273

p =  4.0

Iris Datensatz
c: 3.6358116494468793

Ex

In [2]:
import pandas as pd

df_c = pd.DataFrame(rows_c)
df_c[["scale_factor", "c"]] = df_c[["scale_factor", "c"]].round(4)
df_c = df_c.sort_values(["dataset", "p"]).reset_index(drop=True)
display(df_c)

out_csv = "Tabellen/Bestimmung_c_symmetrisch_p_groesser_2.csv"
df_c.to_csv(out_csv, index=False)
print(f"CSV gespeichert: {out_csv}")

,dataset,p,d,scale_factor,c
0,Covertype,2.3,55,1.8241,62.6160
1,Covertype,2.5,55,2.7233,578.3221
2,Covertype,3.0,55,7.4162,2055.2367
3,Covertype,3.5,55,20.1963,2096.9814
4,Covertype,4.0,55,55.0000,104818.6909
5,Example2D,2.3,3,1.1791,1.1586
6,Example2D,2.5,3,1.3161,1.2763
7,Example2D,3.0,3,1.7321,1.5952
8,Example2D,3.5,3,2.2795,1.9874
9,Example2D,4.0,3,3.0000,2.4646


CSV gespeichert: Tabellen/Bestimmung_c_symmetrisch_p_groesser_2.csv


In [8]:
import numpy as np
import pandas as pd
from efficient_probit_regression.sampling import calculate_lewis_weights_exact, compute_leverage_scores, to_density

# Laden der notwendigen Datensaetze
%store -r Iris
%store -r Webspam
%store -r KDDCup
%store -r Covertype
%store -r Example2D

print("Vergleich von 0.5 * min(d^(p/2-1) * l_2, l_p) <= d^(p/2-1) * lewis <= 2 * max(d^(p/2-1) * l_2, l_p):")
print("")

p = [2.3, 2.5, 3.0, 3.5, 4.0]
LEWIS_ITERATIONS = 20
rows_cmp = []

datasets = {
    "Iris": Iris,
    "Example2D": Example2D,
    "Webspam": Webspam,
    "Covertype": Covertype,
    "KDDCup": KDDCup,
}


def scale_factor_d_p_over_2_minus_1(d: int, p_val: float):
    return float(d ** (p_val / 2.0 - 1.0))


def calculate_scaled_scores(X, p_val):
    d = X.shape[1]
    scale_factor = scale_factor_d_p_over_2_minus_1(d, p_val)
    l2_scores = compute_leverage_scores(X, p=2.0, fast_approx=False)
    lp_scores = compute_leverage_scores(X, p=p_val, fast_approx=False)
    scaled_l2_scores = scale_factor * l2_scores
    l2_prob = to_density(l2_scores)
    lp_prob = to_density(lp_scores)
    return l2_scores, lp_scores, scaled_l2_scores, l2_prob, lp_prob, d, scale_factor


def process_dataset(dataset_name, X, p_val, rows_cmp, T=LEWIS_ITERATIONS):
    print(f"{dataset_name} Datensatz")
    print("")

    l2_scores, lp_scores, scaled_l2_scores, l2_prob, lp_prob, d, scale_factor = calculate_scaled_scores(X, p_val)
    lewis_weights = calculate_lewis_weights_exact(X, p=p_val, T=T)
    scaled_lewis_weights = scale_factor * lewis_weights
    lewis_prob = to_density(lewis_weights)

    print("Vergleich von Scores")
    min_scores = np.minimum(scaled_l2_scores, lp_scores)
    max_scores = np.maximum(scaled_l2_scores, lp_scores)
    compare_scores = (scaled_lewis_weights >= min_scores * 0.5) & (scaled_lewis_weights <= max_scores * 2.0)
    true_scores = int(np.sum(compare_scores))
    false_scores = int(len(compare_scores) - true_scores)
    ratio_true_scores = round(true_scores / len(compare_scores), 4) * 100

    print("Anzahl von TRUE:", true_scores, "Anzahl von FALSE:", false_scores)
    print(ratio_true_scores, "% der Werte sind TRUE")
    print("")

    print("Vergleich von normierten Verteilungen")
    min_prob = np.minimum(l2_prob, lp_prob)
    max_prob = np.maximum(l2_prob, lp_prob)
    compare_prob = (lewis_prob >= min_prob * 0.5) & (lewis_prob <= max_prob * 2.0)
    true_prob = int(np.sum(compare_prob))
    false_prob = int(len(compare_prob) - true_prob)
    ratio_true_prob = round(true_prob / len(compare_prob), 4) * 100

    print("Anzahl von TRUE:", true_prob, "Anzahl von FALSE:", false_prob)
    print(ratio_true_prob, "% der Werte sind TRUE")
    print("")

    rows_cmp.append({
        "dataset": dataset_name,
        "p": float(p_val),
        "d": int(d),
        "scale_factor": float(scale_factor),
        "true_scores": true_scores,
        "false_scores": false_scores,
        "true_scores_percent": ratio_true_scores,
        "true_prob": true_prob,
        "false_prob": false_prob,
        "true_prob_percent": ratio_true_prob,
    })


for p_val in p:
    print("p = ", p_val)
    print("")
    for dataset_name, ds in datasets.items():
        process_dataset(dataset_name, ds.X, p_val, rows_cmp)


Vergleich von 0.5 * min(d^(p/2-1) * l_2, l_p) <= d^(p/2-1) * lewis <= 2 * max(d^(p/2-1) * l_2, l_p):

p =  2.3

Iris Datensatz

Vergleich von Scores
Anzahl von TRUE: 150 Anzahl von FALSE: 0
100.0 % der Werte sind TRUE

Vergleich von normierten Verteilungen
Anzahl von TRUE: 150 Anzahl von FALSE: 0
100.0 % der Werte sind TRUE

Example2D Datensatz

Vergleich von Scores
Anzahl von TRUE: 175 Anzahl von FALSE: 0
100.0 % der Werte sind TRUE

Vergleich von normierten Verteilungen
Anzahl von TRUE: 175 Anzahl von FALSE: 0
100.0 % der Werte sind TRUE

Webspam Datensatz

Vergleich von Scores
Anzahl von TRUE: 350000 Anzahl von FALSE: 0
100.0 % der Werte sind TRUE

Vergleich von normierten Verteilungen
Anzahl von TRUE: 349994 Anzahl von FALSE: 6
100.0 % der Werte sind TRUE

Covertype Datensatz

Vergleich von Scores
Anzahl von TRUE: 580989 Anzahl von FALSE: 23
100.0 % der Werte sind TRUE

Vergleich von normierten Verteilungen
Anzahl von TRUE: 580988 Anzahl von FALSE: 24
100.0 % der Werte sind TRUE

K

In [9]:
df_cmp = pd.DataFrame(rows_cmp).sort_values(["dataset", "p"]).reset_index(drop=True)

float_cols = df_cmp.select_dtypes(include=["float"]).columns
df_cmp[float_cols] = df_cmp[float_cols].round(4)

display(df_cmp)

out_csv = "Tabellen/Vergleich_minmax_l2_lp_vs_lewis_p_groesser_2.csv"
df_cmp.to_csv(out_csv, index=False, float_format="%.4f")
print(f"CSV gespeichert: {out_csv}")


,dataset,p,d,scale_factor,true_scores,false_scores,true_scores_percent,true_prob,false_prob,true_prob_percent
0,Covertype,2.3,55,1.8241,580989,23,100.00,580988,24,100.00
1,Covertype,2.5,55,2.7233,580989,23,100.00,580990,22,100.00
2,Covertype,3.0,55,7.4162,580956,56,99.99,580963,49,99.99
3,Covertype,3.5,55,20.1963,580880,132,99.98,578107,2905,99.50
4,Covertype,4.0,55,55.0000,0,581012,0.00,580399,613,99.89
5,Example2D,2.3,3,1.1791,175,0,100.00,175,0,100.00
6,Example2D,2.5,3,1.3161,175,0,100.00,175,0,100.00
7,Example2D,3.0,3,1.7321,175,0,100.00,175,0,100.00
8,Example2D,3.5,3,2.2795,175,0,100.00,175,0,100.00
9,Example2D,4.0,3,3.0000,0,175,0.00,175,0,100.00


CSV gespeichert: Tabellen/Vergleich_minmax_l2_lp_vs_lewis_p_groesser_2.csv


In [10]:
# Untersuche die maximale Abweichung von scaled_lewis zu max(scaled_l2, l_p)
import numpy as np
import pandas as pd
from efficient_probit_regression.sampling import calculate_lewis_weights_exact, compute_leverage_scores, to_density

# Laden der notwendigen Datensaetze
%store -r Iris
%store -r Webspam
%store -r KDDCup
%store -r Covertype
%store -r Example2D

print("Bestimme die maximale Abweichung von d^(p/2-1) * lewis zu max(d^(p/2-1) * l_2, l_p):")
print("")

p = [2.3, 2.5, 3.0, 3.5, 4.0]
LEWIS_ITERATIONS = 20
EPS = 1e-300
rows_cmp = []

datasets = {
    "Iris": Iris,
    "Example2D": Example2D,
    "Webspam": Webspam,
    "Covertype": Covertype,
    "KDDCup": KDDCup,
}


def scale_factor_d_p_over_2_minus_1(d: int, p_val: float):
    return float(d ** (p_val / 2.0 - 1.0))


def calculate_scaled_scores(X, p_val):
    d = X.shape[1]
    scale_factor = scale_factor_d_p_over_2_minus_1(d, p_val)
    l2_scores = compute_leverage_scores(X, p=2.0, fast_approx=False)
    lp_scores = compute_leverage_scores(X, p=p_val, fast_approx=False)
    scaled_l2_scores = scale_factor * l2_scores
    l2_prob = to_density(l2_scores)
    lp_prob = to_density(lp_scores)
    return l2_scores, lp_scores, scaled_l2_scores, l2_prob, lp_prob, d, scale_factor


def process_dataset(dataset_name, X, p_val, rows_cmp, T=LEWIS_ITERATIONS):
    print(f"{dataset_name} Datensatz")
    print("")

    l2_scores, lp_scores, scaled_l2_scores, l2_prob, lp_prob, d, scale_factor = calculate_scaled_scores(X, p_val)
    lewis_weights = calculate_lewis_weights_exact(X, p=p_val, T=T)
    scaled_lewis_weights = scale_factor * lewis_weights
    lewis_prob = to_density(lewis_weights)

    print("Vergleich von Scores")
    max_scores = np.maximum(scaled_l2_scores, lp_scores)
    max_scores_ratio = round(float(np.max(scaled_lewis_weights / np.maximum(max_scores, EPS))), 4)
    print("scaled_lewis / max(scaled_l2, l_p) =", max_scores_ratio)
    print("")

    print("Vergleich von normierten Verteilungen")
    max_prob = np.maximum(l2_prob, lp_prob)
    max_prob_ratio = round(float(np.max(lewis_prob / np.maximum(max_prob, EPS))), 4)
    print("lewis_prob / max(l_2_prob, l_p_prob) =", max_prob_ratio)
    print("")

    rows_cmp.append({
        "dataset": dataset_name,
        "p": float(p_val),
        "d": int(d),
        "scale_factor": float(scale_factor),
        "max_scores_ratio": max_scores_ratio,
        "max_prob_ratio": max_prob_ratio,
    })


for p_val in p:
    print("p = ", p_val)
    print("")
    for dataset_name, ds in datasets.items():
        process_dataset(dataset_name, ds.X, p_val, rows_cmp)


Bestimme die maximale Abweichung von d^(p/2-1) * lewis zu max(d^(p/2-1) * l_2, l_p):

p =  2.3

Iris Datensatz

Vergleich von Scores
scaled_lewis / max(scaled_l2, l_p) = 1.122

Vergleich von normierten Verteilungen
lewis_prob / max(l_2_prob, l_p_prob) = 1.1021

Example2D Datensatz

Vergleich von Scores
scaled_lewis / max(scaled_l2, l_p) = 1.1639

Vergleich von normierten Verteilungen
lewis_prob / max(l_2_prob, l_p_prob) = 1.1639

Webspam Datensatz

Vergleich von Scores
scaled_lewis / max(scaled_l2, l_p) = 1.6863

Vergleich von normierten Verteilungen
lewis_prob / max(l_2_prob, l_p_prob) = 1.6863

Covertype Datensatz

Vergleich von Scores
scaled_lewis / max(scaled_l2, l_p) = 80.2697

Vergleich von normierten Verteilungen
lewis_prob / max(l_2_prob, l_p_prob) = 79.2809

KDDCup Datensatz

Vergleich von Scores
scaled_lewis / max(scaled_l2, l_p) = 1.442

Vergleich von normierten Verteilungen
lewis_prob / max(l_2_prob, l_p_prob) = 1.4168

p =  2.5

Iris Datensatz

Vergleich von Scores
scaled_

In [11]:
df_cmp = pd.DataFrame(rows_cmp).sort_values(["dataset", "p"]).reset_index(drop=True)

float_cols = df_cmp.select_dtypes(include=["float"]).columns
df_cmp[float_cols] = df_cmp[float_cols].round(4)

display(df_cmp)

out_csv = "Tabellen/Maximale_Abweichung_lewis_max_l2_lp_p_groesser_2.csv"
df_cmp.to_csv(out_csv, index=False, float_format="%.4f")
print(f"CSV gespeichert: {out_csv}")


,dataset,p,d,scale_factor,max_scores_ratio,max_prob_ratio
0,Covertype,2.3,55,1.8241,8.026970e+01,79.2809
1,Covertype,2.5,55,2.7233,1.185985e+02,115.3715
2,Covertype,3.0,55,7.4162,7.433805e+02,702.8366
3,Covertype,3.5,55,20.1963,3.225454e+02,273.4841
4,Covertype,4.0,55,55.0000,1.387351e+09,51477.9509
5,Example2D,2.3,3,1.1791,1.163900e+00,1.1639
6,Example2D,2.5,3,1.3161,1.279200e+00,1.1964
7,Example2D,3.0,3,1.7321,1.585800e+00,1.5858
8,Example2D,3.5,3,2.2795,1.936800e+00,1.6186
9,Example2D,4.0,3,3.0000,8.960860e+01,1.6812


CSV gespeichert: Tabellen/Maximale_Abweichung_lewis_max_l2_lp_p_groesser_2.csv


In [12]:
# Bestimmung von c = max(lewis_prob / scaled_l2_lp_prob)
import numpy as np
from efficient_probit_regression.sampling import calculate_lewis_weights_exact, compute_leverage_scores, to_density

# Laden der notwendigen Datensaetze
%store -r Iris
%store -r Webspam
%store -r KDDCup
%store -r Covertype
%store -r Example2D

print("Bestimmung von c = max(lewis_prob / scaled_l2_lp_prob)")
print("")

p = [2.3, 2.5, 3.0, 3.5, 4.0]
LEWIS_ITERATIONS = 20
EPS = 1e-300
rows_c = []

datasets = {
    "Iris": Iris,
    "Example2D": Example2D,
    "Webspam": Webspam,
    "Covertype": Covertype,
    "KDDCup": KDDCup,
}


def scale_factor_d_p_over_2_minus_1(d: int, p_val: float):
    return float(d ** (p_val / 2.0 - 1.0))


def calculate_scaled_l2_lp_scores(X, p_val):
    d = X.shape[1]
    scale_factor = scale_factor_d_p_over_2_minus_1(d, p_val)
    l2_scores = compute_leverage_scores(X, p=2.0, fast_approx=False)
    lp_scores = compute_leverage_scores(X, p=p_val, fast_approx=False)
    scaled_scores = scale_factor * l2_scores + lp_scores
    scaled_prob = to_density(scaled_scores)
    return scaled_scores, scaled_prob, d, scale_factor


for p_val in p:
    print("p = ", p_val)
    print("")

    for dataset_name, ds in datasets.items():
        print(f"{dataset_name} Datensatz")
        lewis_weights = calculate_lewis_weights_exact(ds.X, p=p_val, T=LEWIS_ITERATIONS)
        lewis_prob = to_density(lewis_weights)
        scaled_scores, scaled_prob, d, scale_factor = calculate_scaled_l2_lp_scores(ds.X, p_val)

        c_value = float(np.max(lewis_prob / np.maximum(scaled_prob, EPS)))
        print("c:", c_value)
        rows_c.append({
            "dataset": dataset_name,
            "p": float(p_val),
            "d": int(d),
            "scale_factor": float(scale_factor),
            "c": float(c_value),
        })
        print("")


Bestimmung von c = max(lewis_prob / scaled_l2_lp_prob)

p =  2.3

Iris Datensatz
c: 1.1263906940155335

Example2D Datensatz
c: 1.1663515654139034

Webspam Datensatz
c: 1.6839467371493098

Covertype Datensatz
c: 131.16679864657448

KDDCup Datensatz
c: 1.4379335363464643

p =  2.5

Iris Datensatz
c: 1.207061385055249

Example2D Datensatz
c: 1.2881611735106677

Webspam Datensatz
c: 2.350366959549758

Covertype Datensatz
c: 71.83753642431185

KDDCup Datensatz
c: 1.803280836264973

p =  3.0

Iris Datensatz
c: 1.4266964824683672

Example2D Datensatz
c: 1.5867725410350066

Webspam Datensatz
c: 4.96420749507083

Covertype Datensatz
c: 591.3018945753536

KDDCup Datensatz
c: 2.959174014084868

p =  3.5

Iris Datensatz
c: 1.6562123647164264

Example2D Datensatz
c: 1.9119371825155185

Webspam Datensatz
c: 9.15267615069895

Covertype Datensatz
c: 600.1312853779376

KDDCup Datensatz
c: 4.509024274746808

p =  4.0

Iris Datensatz
c: 1.8937491945352707

Example2D Datensatz
c: 2.258800384615893

Webspa

In [13]:
import pandas as pd

df_c = pd.DataFrame(rows_c)
df_c[["scale_factor", "c"]] = df_c[["scale_factor", "c"]].round(4)
df_c = df_c.sort_values(["dataset", "p"]).reset_index(drop=True)
display(df_c)

out_csv = "Tabellen/Bestimmung_c_max_p_groesser_2.csv"
df_c.to_csv(out_csv, index=False)
print(f"CSV gespeichert: {out_csv}")


,dataset,p,d,scale_factor,c
0,Covertype,2.3,55,1.8241,131.1668
1,Covertype,2.5,55,2.7233,71.8375
2,Covertype,3.0,55,7.4162,591.3019
3,Covertype,3.5,55,20.1963,600.1313
4,Covertype,4.0,55,55.0000,2573.1901
5,Example2D,2.3,3,1.1791,1.1664
6,Example2D,2.5,3,1.3161,1.2882
7,Example2D,3.0,3,1.7321,1.5868
8,Example2D,3.5,3,2.2795,1.9119
9,Example2D,4.0,3,3.0000,2.2588


CSV gespeichert: Tabellen/Bestimmung_c_max_p_groesser_2.csv
